# 06 — Mishra/Sarkhi Lightweight CNN Baseline

This notebook reproduces the compact TensorFlow/Keras baseline used in:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It is based on the final source notebook:

> **Mishra/Sarkhi Lightweight CNN Baseline — Faithful Fast GPU Reconstruction**

## Scope

- reuse the controlled train/validation/test splits from Notebook 01;
- preprocess images to 128 × 128 RGB tensors normalized to [0,1];
- train the compact three-block CNN;
- evaluate the final trained model on the untouched test split;
- save model, metrics, predictions, curves, confusion matrix, and configuration files.

The dataset is not re-split here.

## Source-of-Truth Configuration

The settings below are taken directly from the final experiment notebook and should be used when updating the manuscript.

| Parameter | Value |
|---|---|
| Input size | 128 × 128 × 3 |
| Batch size | 256 |
| Epochs | 8 |
| Optimizer | Adam |
| Learning rate | 1 × 10⁻³ |
| Loss | Binary cross-entropy |
| Convolution filters | 8 → 16 → 32 |
| Kernel size | 3 × 3 |
| Padding | Same |
| Pooling | Max pooling, 2 × 2 |
| Feature aggregation | Global average pooling |
| Dense layer | 32 neurons, ReLU |
| Dropout | 0.50 |
| Output | 1 sigmoid neuron |
| Training strategy | Fixed schedule; final-epoch evaluation |
| Checkpoint selection | Not used |
| Learning-rate scheduler | Not used |
| Data augmentation | Not used |
| Random seed | 42 |

## 1. Imports, GPU Configuration, and Reproducibility

In [ ]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["PYTHONHASHSEED"] = "42"

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow version:", tf.__version__)
print("GPU devices:", gpus)

if gpus:
    print(
        "Mixed precision policy:",
        tf.keras.mixed_precision.global_policy(),
    )

## 2. Repository Paths and Hyperparameters

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "Data" / "processed"

MODEL_DIR = PROJECT_ROOT / "Models" / "mishra_lightweight_cnn"
RESULT_DIR = PROJECT_ROOT / "Results" / "mishra_lightweight_cnn"
FIGURES_DIR = RESULT_DIR / "figures"
TABLES_DIR = RESULT_DIR / "tables"
METRICS_DIR = RESULT_DIR / "metrics"

for directory in [
    MODEL_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    METRICS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV = PROCESSED_DATA_DIR / "test.csv"

IMG_SIZE = 128
BATCH_SIZE = 256
EPOCHS = 8
LEARNING_RATE = 1e-3
CLASSIFICATION_THRESHOLD = 0.50

print("Project root:", PROJECT_ROOT)
print("Input size  :", IMG_SIZE)
print("Batch size  :", BATCH_SIZE)
print("Epochs      :", EPOCHS)

## 3. Load the Controlled Dataset Splits

In [ ]:
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Run Notebook 01 first."
        )

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    dataframe = pd.read_csv(path).copy()

    if "image_path" not in dataframe.columns and "image" in dataframe.columns:
        dataframe = dataframe.rename(columns={"image": "image_path"})

    required = {"image_path", "label"}
    missing = required - set(dataframe.columns)

    if missing:
        raise ValueError(
            f"{split_name} split is missing columns: {sorted(missing)}"
        )

    dataframe["image_path"] = (
        dataframe["image_path"].astype(str).str.strip()
    )
    dataframe["label"] = (
        dataframe["label"].astype(str).str.strip().str.lower()
    )

    if "label_id" not in dataframe.columns:
        dataframe["label_id"] = dataframe["label"].map(
            {"benign": 0, "malicious": 1}
        )

    dataframe["label_id"] = pd.to_numeric(
        dataframe["label_id"],
        errors="coerce",
    )

    if dataframe["label_id"].isna().any():
        raise ValueError(
            f"{split_name} split contains invalid labels."
        )

    dataframe["label_id"] = dataframe["label_id"].astype(int)

    exists_mask = dataframe["image_path"].map(
        lambda path_value: (
            PROJECT_ROOT / path_value
        ).exists()
    )

    missing_count = int((~exists_mask).sum())

    if missing_count:
        raise FileNotFoundError(
            f"{split_name} split contains {missing_count} missing image files."
        )

    dataframe["absolute_image_path"] = dataframe[
        "image_path"
    ].map(
        lambda path_value: str(
            (PROJECT_ROOT / path_value).resolve()
        )
    )

    return dataframe.reset_index(drop=True)

train_df = load_split(TRAIN_CSV, "train")
val_df = load_split(VAL_CSV, "validation")
test_df = load_split(TEST_CSV, "test")

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain distribution:")
print(train_df["label"].value_counts())

## 4. TensorFlow Image Pipeline

Images are decoded as RGB, resized with bilinear interpolation and antialiasing,
converted to `float32`, and normalized to [0,1].

No augmentation is applied.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
PATH_COLUMN = "absolute_image_path"
LABEL_COLUMN = "label_id"

def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False,
    )
    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True,
    )

    image = tf.cast(image, tf.float32) / 255.0
    label = tf.cast(label, tf.float32)

    return image, label

def make_dataset(
    dataframe: pd.DataFrame,
    training: bool = False,
    cache: bool = False,
):
    image_paths = dataframe[PATH_COLUMN].astype(str).to_numpy()
    labels = dataframe[LABEL_COLUMN].astype(
        "float32"
    ).to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    options = tf.data.Options()
    options.experimental_deterministic = not training
    dataset = dataset.with_options(options)

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 30000),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training,
    )

    if cache:
        dataset = dataset.cache()

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False,
    )
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = make_dataset(
    train_df,
    training=True,
    cache=False,
)
val_ds = make_dataset(
    val_df,
    training=False,
    cache=True,
)
test_ds = make_dataset(
    test_df,
    training=False,
    cache=True,
)

sample_images, sample_labels = next(iter(train_ds))

print("Image batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print(
    "Image value range:",
    float(tf.reduce_min(sample_images)),
    float(tf.reduce_max(sample_images)),
)

## 5. Build the Compact Lightweight CNN

In [ ]:
def build_mishra_lightweight_cnn(input_shape):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=input_shape),

            tf.keras.layers.Conv2D(
                8,
                kernel_size=3,
                activation="relu",
                padding="same",
            ),
            tf.keras.layers.MaxPooling2D(
                pool_size=(2, 2)
            ),

            tf.keras.layers.Conv2D(
                16,
                kernel_size=3,
                activation="relu",
                padding="same",
            ),
            tf.keras.layers.MaxPooling2D(
                pool_size=(2, 2)
            ),

            tf.keras.layers.Conv2D(
                32,
                kernel_size=3,
                activation="relu",
                padding="same",
            ),
            tf.keras.layers.MaxPooling2D(
                pool_size=(2, 2)
            ),

            tf.keras.layers.GlobalAveragePooling2D(),

            tf.keras.layers.Dense(
                32,
                activation="relu",
            ),
            tf.keras.layers.Dropout(
                0.50,
                seed=SEED,
            ),

            tf.keras.layers.Dense(
                1,
                activation="sigmoid",
                dtype="float32",
            ),
        ]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(
                name="precision"
            ),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )

    return model

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = build_mishra_lightweight_cnn(
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

model.summary()

## 6. Fixed-Schedule Training

The source experiment evaluates the final trained model directly. It does not use
best-checkpoint selection, early stopping, or a learning-rate scheduler.

In [ ]:
callbacks = [
    tf.keras.callbacks.CSVLogger(
        str(TABLES_DIR / "training_history_logger.csv"),
        append=False,
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

training_start = time.perf_counter()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

training_time_seconds = (
    time.perf_counter() - training_start
)

history_df = pd.DataFrame(history.history)
HISTORY_CSV = TABLES_DIR / "training_history.csv"
history_df.to_csv(HISTORY_CSV, index=False)

FINAL_MODEL_PATH = (
    MODEL_DIR / "final_mishra_lightweight_cnn.keras"
)
model.save(FINAL_MODEL_PATH)

print("Training time:", training_time_seconds)
print("Saved history:", HISTORY_CSV)
print("Saved model  :", FINAL_MODEL_PATH)

## 7. Training Curves

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    history.history["accuracy"],
    label="Training Accuracy",
)
plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Mishra Lightweight CNN Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "accuracy_curve.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(
    history.history["loss"],
    label="Training Loss",
)
plt.plot(
    history.history["val_loss"],
    label="Validation Loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Mishra Lightweight CNN Loss")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "loss_curve.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. Test Evaluation and Inference Time

In [ ]:
inference_start = time.perf_counter()
y_prob = model.predict(test_ds).ravel()
inference_time_seconds = (
    time.perf_counter() - inference_start
)

y_true = test_df[LABEL_COLUMN].astype(int).to_numpy()
y_pred = (
    y_prob >= CLASSIFICATION_THRESHOLD
).astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)
recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)
f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)
roc_auc = roc_auc_score(y_true, y_prob)
average_precision = average_precision_score(
    y_true,
    y_prob,
)

metrics = {
    "model": "Mishra/Sarkhi Compact Reconstructed Lightweight CNN",
    "framework": "TensorFlow/Keras",
    "image_size": f"{IMG_SIZE}x{IMG_SIZE}",
    "batch_size": BATCH_SIZE,
    "epochs_configured": EPOCHS,
    "epochs_completed": len(
        history.history["loss"]
    ),
    "learning_rate": LEARNING_RATE,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "split": "70/15/15 stratified, seed 42",
    "test_accuracy": accuracy,
    "test_precision": precision,
    "test_recall": recall,
    "test_f1_score": f1,
    "test_roc_auc": roc_auc,
    "test_average_precision": average_precision,
    "training_time_seconds": training_time_seconds,
    "inference_time_seconds": inference_time_seconds,
    "average_inference_time_per_image_seconds": (
        inference_time_seconds / len(test_df)
    ),
}

metrics_df = pd.DataFrame([metrics])
METRICS_CSV = (
    METRICS_DIR / "mishra_lightweight_cnn_metrics.csv"
)
metrics_df.to_csv(METRICS_CSV, index=False)

predictions_df = test_df[
    ["image_path", "label", "label_id"]
].copy()
predictions_df["predicted_label_id"] = y_pred
predictions_df["probability_malicious"] = y_prob
predictions_df.to_csv(
    TABLES_DIR / "test_predictions.csv",
    index=False,
)

display(metrics_df.T)

## 9. Classification Report and Confusion Matrix

In [ ]:
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=["Benign", "Malicious"],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(
    TABLES_DIR / "classification_report.csv"
)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Benign", "Malicious"],
        zero_division=0,
    )
)

matrix = confusion_matrix(y_true, y_pred)

matrix_df = pd.DataFrame(
    matrix,
    index=["Actual Benign", "Actual Malicious"],
    columns=["Predicted Benign", "Predicted Malicious"],
)
matrix_df.to_csv(
    TABLES_DIR / "confusion_matrix.csv"
)

display_object = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["Benign", "Malicious"],
)
display_object.plot(values_format="d")
plt.title("Mishra Lightweight CNN Confusion Matrix")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 10. ROC and Precision–Recall Curves

In [ ]:
false_positive_rate, true_positive_rate, _ = (
    roc_curve(y_true, y_prob)
)

pd.DataFrame(
    {
        "false_positive_rate": false_positive_rate,
        "true_positive_rate": true_positive_rate,
    }
).to_csv(
    TABLES_DIR / "roc_curve_points.csv",
    index=False,
)

plt.figure(figsize=(7, 5))
plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"AUC = {roc_auc:.4f}",
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mishra Lightweight CNN ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "roc_curve.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

pr_precision, pr_recall, _ = precision_recall_curve(
    y_true,
    y_prob,
)

pd.DataFrame(
    {
        "precision": pr_precision,
        "recall": pr_recall,
    }
).to_csv(
    TABLES_DIR / "precision_recall_curve_points.csv",
    index=False,
)

plt.figure(figsize=(7, 5))
plt.plot(
    pr_recall,
    pr_precision,
    label=f"AP = {average_precision:.4f}",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    "Mishra Lightweight CNN Precision–Recall Curve"
)
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Save Hyperparameters and Run Summary

In [ ]:
hyperparameters = {
    "Model": "Mishra/Sarkhi compact reconstructed lightweight CNN",
    "Framework": "TensorFlow/Keras",
    "Input size": f"{IMG_SIZE} x {IMG_SIZE} x 3",
    "Preprocessing": (
        "Decode, bilinear resize with antialiasing, "
        "float conversion, normalization to [0,1]"
    ),
    "Split": (
        "70% training, 15% validation, 15% testing"
    ),
    "Random seed": SEED,
    "Batch size": BATCH_SIZE,
    "Epochs": EPOCHS,
    "Optimizer": "Adam",
    "Learning rate": LEARNING_RATE,
    "Loss": "Binary cross-entropy",
    "Convolution filters": "8 -> 16 -> 32",
    "Kernel size": "3 x 3",
    "Padding": "Same",
    "Pooling": "Max pooling 2 x 2",
    "Feature aggregation": "GlobalAveragePooling2D",
    "Dense layer": "32 neurons, ReLU",
    "Dropout": 0.50,
    "Output": "1 sigmoid neuron",
    "Data augmentation": "None",
    "Checkpoint selection": "None",
    "Early stopping": "None",
    "Learning-rate scheduler": "None",
    "Training strategy": (
        "Fixed schedule; final-epoch evaluation"
    ),
    "Callbacks": "CSVLogger, TerminateOnNaN",
}

hyperparameter_df = pd.DataFrame(
    list(hyperparameters.items()),
    columns=["Parameter", "Value"],
)
hyperparameter_df.to_csv(
    TABLES_DIR / "hyperparameter_table.csv",
    index=False,
)

with (
    METRICS_DIR / "run_summary.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        {**hyperparameters, **metrics},
        file,
        indent=4,
    )

display(hyperparameter_df)

## 12. Final Validation

In [ ]:
assert len(y_true) == len(test_df)
assert len(y_pred) == len(test_df)
assert len(y_prob) == len(test_df)
assert not np.isnan(y_prob).any()
assert set(np.unique(y_true)).issubset({0, 1})
assert set(np.unique(y_pred)).issubset({0, 1})

print("=" * 72)
print("MISHRA LIGHTWEIGHT CNN BASELINE COMPLETED")
print("=" * 72)
display(metrics_df)
print("Model  :", FINAL_MODEL_PATH)
print("Results:", RESULT_DIR)
print("=" * 72)